In [ ]:
# Colab setup. Use Runtime -> Change runtime type -> GPU before running.
import locale
import subprocess
import sys

locale.getpreferredencoding = lambda: "UTF-8"
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "unsloth", "trl", "datasets", "accelerate", "peft", "bitsandbytes",
])


In [ ]:
from pathlib import Path

from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTConfig, SFTTrainer

if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. In Colab, choose Runtime -> Change runtime type -> GPU.")

if not Path("datasets/pascal_alpaca.json").exists():
    from google.colab import files
    print("Upload datasets/pascal_alpaca.json")
    files.upload()

max_seq_length = 2048
dtype = None
load_in_4bit = True
model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

dataset = load_dataset("json", data_files="datasets/pascal_alpaca.json", split="train")
INSTRUCTION = dataset[0]["instruction"]

def format_example(example):
    messages = [
        {"role": "user", "content": f"{example['instruction']}\n\nPlayer: {example['input']}"},
        {"role": "assistant", "content": example["output"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

dataset = dataset.map(format_example, remove_columns=dataset.column_names)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        output_dir="pascal_unsloth_mistral_lora_en",
        dataset_text_field="text",
        max_length=max_seq_length,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim="adamw_8bit",
        logging_steps=10,
        save_strategy="epoch",
        report_to="none",
    ),
)

trainer.train()

adapter_dir = "pascal_unsloth_mistral_lora_en"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"Saved LoRA adapter to {adapter_dir}")

FastLanguageModel.for_inference(model)

def chat(player_input, max_new_tokens=80):
    messages = [{"role": "user", "content": f"{INSTRUCTION}\n\nPlayer: {player_input}"}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        repetition_penalty=1.05,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True).strip()

for prompt in [
    "The clouds looked weird today.",
    "I found a scallop while diving.",
    "I feel like everyone on the island is busy except me.",
    "My snack fell on the floor and I got philosophical about it.",
]:
    print("Player:", prompt)
    print("Pascal:", chat(prompt))
    print()

# Optional merged export for deployment. This uses more RAM/VRAM.
# model.save_pretrained_merged("pascal_unsloth_mistral_lora_en_merged", tokenizer, save_method="merged_16bit")
